# Proyecto ETL con arquitectura Medallón

Este cuaderno carga los resultados reales y muestra los DataFrames de cada capa. Ejecuta las celdas en orden desde la raíz del proyecto. Para usar JupyterLab, instala opcionalmente `python -m pip install jupyterlab` y abre con `python -m jupyterlab proyecto_medallon.ipynb`. El ETL por consola no necesita Jupyter.

2025 solo incluye ICFES 20251: no representa un año completo.

## 1. Cargar los resultados

Se preservan los ceros iniciales de DIVIPOLA y los valores nulos. En la carpeta original se lee la última ejecución; en la entrega ZIP se lee `resultados/`.

In [1]:
from pathlib import Path
from etl_medallon import cargar_dataframes, ejecutar_etl

datos = (cargar_dataframes(run_dir="resultados")
         if Path("resultados/report.json").exists() else cargar_dataframes())
for nombre, tabla in datos.items():
    print(f"{nombre}: {tabla.shape[0]:,} filas y {tabla.shape[1]} columnas")

df_icfes: 3,982 filas y 6 columnas
df_crc: 17,276 filas y 4 columnas
df_dane: 4,486 filas y 6 columnas
df_unificado: 4,492 filas y 17 columnas
df_final: 3,427 filas y 13 columnas
df_resumen_anual: 4 filas y 5 columnas
df_cobertura: 4,492 filas y 6 columnas
df_balance: 9 filas y 4 columnas
df_bronze_catalogo: 9 filas y 5 columnas


## 2. Bronze: archivos originales y trazabilidad

Cada fuente conserva su propio formato. El catálogo registra los archivos y sus huellas SHA-256. No se concatenan estudiantes con conexiones u hogares porque representan unidades diferentes. Las rutas del manifiesto describen el equipo de la ejecución original.

In [2]:
df_bronze_catalogo = datos["df_bronze_catalogo"]
print(df_bronze_catalogo[["type", "bytes", "sha256"]].to_string(index=False))

 type     bytes                                                           sha256
icfes  27123682 1f8eabad427e23a7982576c8c5529c0275a674e5f2418bc82a024b9b344e51dd
icfes 406051795 7dc0b4c8818ca160c871e4ee3bb31296fef4a35cd3177891885a3e4090d97f8e
icfes  28076126 acd1605cd2e1134ba94817a7f12f454f10f67e5a6e7b59c2a2605d271de3cc04
icfes 418054810 7bf1c9226b5949a304d3665a4f21ceeada7909a28798d3825f7b74ced03ed9ae
icfes  30261828 e832de9eaa22796969e94a0dabb455a04542d41317abf1a0f59836a000e73db8
icfes 410113425 f0fdbbe240b520469b2e79293587343e6bf6ce3d2812a448b5bc23e5760a6454
icfes  30630005 d4f902d9a47233a7549ff8d411618bebea2626b820476c31bfa8e18efb2ee484
  crc 441330430 2fa895b0bb5d19c946c5f9c6f060e9de4036281a27c29c7e5a35243600d6bb5e
 dane    670894 597dd8ff4745b456fe0eafceb405ad2d8a549d749212adc51b6482a73d2ad269


## 3. Silver: DataFrames unificados por fuente

`df_icfes` reúne los siete TXT a nivel municipio-periodo. `df_crc` suma accesos residenciales por municipio-trimestre. `df_dane` transforma las columnas de años a filas municipio-año. Los datos inválidos quedan identificados en los reportes de calidad.

In [3]:
df_icfes = datos["df_icfes"]
df_crc = datos["df_crc"]
df_dane = datos["df_dane"]
for nombre in ["df_icfes", "df_crc", "df_dane"]:
    print("\n" + nombre)
    print(datos[nombre].head(3).to_string(index=False))


df_icfes
codigo_municipio periodo  suma_puntaje  estudiantes  anio  puntaje_promedio
           05001   20221        138934          553  2022        251.236890
           05001   20222       7060012        28185  2022        250.488274
           05001   20231        116759          436  2023        267.795872

df_crc
codigo_municipio  anio  trimestre  accesos
           05001  2022          1   647782
           05001  2022          2   656454
           05001  2022          3   662809

df_dane
codigo_municipio municipio departamento  anio  hogares  fila_excel
           05001  Medellín    Antioquia  2022   873404          12
           05001  Medellín    Antioquia  2023   883067          12
           05001  Medellín    Antioquia  2024   894459          12


## 4. Gold: unificación de las tres fuentes

Antes de cruzar, se lleva ICFES y CRC a municipio-año. El puntaje se pondera por cantidad de registros, y los accesos son el promedio de totales trimestrales. `df_unificado` hace una unión externa: mantiene todas las llaves y conserva los nulos, sin reemplazarlos con ceros.

In [4]:
df_unificado = datos["df_unificado"]
print(df_unificado.head().to_string(index=False))
print("\nTipos de datos:")
print(df_unificado.dtypes.to_string())
print("\nFilas sin las tres fuentes:", int((~df_unificado.integrado).sum()))

codigo_municipio  anio  estudiantes  periodos_icfes  puntaje_global  tiene_icfes  accesos_residenciales  trimestres_crc  tiene_crc municipio departamento  hogares  tiene_dane  integrado  accesos_por_100_hogares  cobertura_temporal_completa  tasa_mayor_100
           05001  2022        28738               2      250.502679         True              661333.25               4       True  Medellín    Antioquia   873404        True       True                75.719054                         True           False
           05001  2023        28501               2      254.030911         True              681446.75               4       True  Medellín    Antioquia   883067        True       True                77.168182                         True           False
           05001  2024        28288               2      256.112062         True              690924.75               4       True  Medellín    Antioquia   894459        True       True                77.244988                      

## 5. Gold: DataFrame final para análisis

`df_final` selecciona municipios-año con las tres fuentes. El indicador es accesos residenciales medios / hogares × 100. No mide el porcentaje de hogares conectados y puede superar 100. La presencia de tres fuentes no garantiza que el año esté completo.

In [5]:
df_final = datos["df_final"]
df_resumen_anual = datos["df_resumen_anual"]
print(df_final.head().to_string(index=False))
print("\nResumen anual:")
print(df_resumen_anual.to_string(index=False))

codigo_municipio  anio  estudiantes  periodos_icfes  puntaje_global  accesos_residenciales  trimestres_crc municipio departamento  hogares  accesos_por_100_hogares  cobertura_temporal_completa  tasa_mayor_100
           05001  2022        28738               2      250.502679              661333.25               4  Medellín    Antioquia   873404                75.719054                         True           False
           05001  2023        28501               2      254.030911              681446.75               4  Medellín    Antioquia   883067                77.168182                         True           False
           05001  2024        28288               2      256.112062              690924.75               4  Medellín    Antioquia   894459                77.244988                         True           False
           05001  2025          474               1      250.259494              745520.75               4  Medellín    Antioquia   904765                82.399380 

## 6. Revisar calidad

El balance registra válidos y rechazados. El reporte de cobertura muestra las llaves sin coincidencia. Tener un solo periodo en un municipio también puede reflejar su calendario escolar; no es automáticamente un error. La ausencia del archivo nacional 20252 sí está confirmada.

In [6]:
df_balance = datos["df_balance"]
df_cobertura = datos["df_cobertura"]
print(df_balance.to_string(index=False))
print("\nEjemplos de llaves no integradas:")
print(df_cobertura.loc[~df_cobertura.integrado].head().to_string(index=False))
assert not df_final.duplicated(["codigo_municipio", "anio"]).any()
assert df_final.codigo_municipio.str.fullmatch(r"[0-9]{5}").all()
assert df_final.hogares.gt(0).all()
assert len(df_final) == int(df_unificado.integrado.sum())
print("\nControles de llave, formato, denominador y selección: correctos.")

                                             archivo  leidos  rechazados  validos
                       ACCESOS_INTERNET_FIJO_2_8.csv 3288251           0  3288251
                           Examen_Saber_11_20221.txt   73795       53746    20049
                           Examen_Saber_11_20222.txt  589183       45477   543706
                           Examen_Saber_11_20231.txt   77555       56903    20652
                           Examen_Saber_11_20232.txt  602093       39179   562914
                           Examen_Saber_11_20241.txt   84072       63442    20630
                           Examen_Saber_11_20242.txt  592436       35138   557298
                           Examen_Saber_11_20251.txt   85678       63885    21793
anexo-proyecciones-hogares-dptal-mpal-2018-2042.xlsx    4492           6     4486

Ejemplos de llaves no integradas:
codigo_municipio  anio  tiene_icfes  tiene_crc  tiene_dane  integrado
           05002  2025        False       True        True      False
     

## 7. Reconstruir todo cuando cambien las fuentes

Configura `config/local.json` siguiendo el README. Descomenta la siguiente celda para ejecutar nuevamente Bronze, Silver y Gold. Las copias de originales están en el equipo local y no van incluidas en el ZIP.

In [7]:
# datos = ejecutar_etl()
# df_unificado = datos["df_unificado"]
# df_final = datos["df_final"]